In [ ]:
from google.colab import files

uploaded = files.upload()

Saving master_model_ready.csv to master_model_ready.csv


In [ ]:
import pandas as pd

df = pd.read_csv("master_model_ready.csv")

print("Shape:", df.shape)
df.head()

Shape: (40, 26)


,location_id,latitude,longitude,elevation_m,rainfall_2026_mm,max_hourly_rainfall_2026_mm,rainfall_2024_mm,max_hourly_rainfall_2024_mm,rainfall_2025_mm,max_hourly_rainfall_2025_mm,...,distance_to_watershed_boundary_m,natural_flow_elevation,natural_flow_in_flow,natural_flow_out_flow,drain_area_sq_km,nearest_natural_flow_distance_m,traffic_level_High,traffic_level_Moderate,traffic_level_Severe,flood_label
0,H01,28.438870,77.005661,221,556.3,11.3,802.2,29.2,559.8,19.8,...,235.5,0,111,576,0.8410,151.9,1,0,0,1
1,H02,28.444206,76.999750,220,556.3,11.3,802.2,29.2,559.8,19.8,...,242.2,0,111,576,0.8410,41.6,0,0,1,1
2,H03,28.454574,77.042674,227,556.3,11.3,802.2,29.2,559.8,19.8,...,1896.4,0,100,185,0.2702,432.0,1,0,0,1
3,H04,28.460984,77.042014,226,556.3,11.3,802.2,29.2,559.8,19.8,...,1535.8,0,387,540,0.7890,70.3,1,0,0,1
4,H05,28.453822,77.045449,229,556.3,11.3,802.2,29.2,559.8,19.8,...,2218.9,0,100,185,0.2702,153.8,1,0,0,1


In [ ]:
import pandas as pd

df = pd.read_csv("master_model_ready.csv")

print("Shape:", df.shape)
df.head()

Shape: (40, 26)


,location_id,latitude,longitude,elevation_m,rainfall_2026_mm,max_hourly_rainfall_2026_mm,rainfall_2024_mm,max_hourly_rainfall_2024_mm,rainfall_2025_mm,max_hourly_rainfall_2025_mm,...,distance_to_watershed_boundary_m,natural_flow_elevation,natural_flow_in_flow,natural_flow_out_flow,drain_area_sq_km,nearest_natural_flow_distance_m,traffic_level_High,traffic_level_Moderate,traffic_level_Severe,flood_label
0,H01,28.438870,77.005661,221,556.3,11.3,802.2,29.2,559.8,19.8,...,235.5,0,111,576,0.8410,151.9,1,0,0,1
1,H02,28.444206,76.999750,220,556.3,11.3,802.2,29.2,559.8,19.8,...,242.2,0,111,576,0.8410,41.6,0,0,1,1
2,H03,28.454574,77.042674,227,556.3,11.3,802.2,29.2,559.8,19.8,...,1896.4,0,100,185,0.2702,432.0,1,0,0,1
3,H04,28.460984,77.042014,226,556.3,11.3,802.2,29.2,559.8,19.8,...,1535.8,0,387,540,0.7890,70.3,1,0,0,1
4,H05,28.453822,77.045449,229,556.3,11.3,802.2,29.2,559.8,19.8,...,2218.9,0,100,185,0.2702,153.8,1,0,0,1


In [ ]:
location_ids = df["location_id"]

X = df.drop(columns=["location_id", "flood_label"])
y = df["flood_label"]

print("Features:", X.shape)
print("Target:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features: (40, 24)
Target: (40,)

Target distribution:
flood_label
1    20
0    20
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

X_train: (32, 24)
X_test : (8, 24)
y_train: (32,)
y_test : (8,)

Training labels:
flood_label
0    16
1    16
Name: count, dtype: int64

Testing labels:
flood_label
0    4
1    4
Name: count, dtype: int64


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000))
])

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.75

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         4
           1       0.67      1.00      0.80         4

    accuracy                           0.75         8
   macro avg       0.83      0.75      0.73         8
weighted avg       0.83      0.75      0.73         8


Confusion Matrix:
[[2 2]
 [0 4]]


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("5-Fold CV Accuracy:")
print(cv_scores)

print("\nMean Accuracy:", cv_scores.mean())
print("Std Dev:", cv_scores.std())

5-Fold CV Accuracy:
[0.625 0.625 0.625 0.75  0.875]

Mean Accuracy: 0.7
Std Dev: 0.1


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)

rf_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Random Forest 5-Fold Accuracy:")
print(rf_scores)

print("\nMean Accuracy:", rf_scores.mean())
print("Std Dev:", rf_scores.std())

Random Forest 5-Fold Accuracy:
[0.625 0.5   0.75  0.75  0.875]

Mean Accuracy: 0.7
Std Dev: 0.1274754878398196


In [ ]:
# Train Random Forest on the full dataset for feature importance
rf_model.fit(X, y)

feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)

normal_duration_sec                 0.104053
natural_flow_in_flow                0.098426
latitude                            0.091987
traffic_duration_sec                0.073350
route_distance_m                    0.072371
distance_to_watershed_boundary_m    0.069888
natural_flow_out_flow               0.058226
congestion_ratio                    0.057911
drain_area_sq_km                    0.041902
longitude                           0.041683
rainfall_2025_mm                    0.040754
traffic_delay_sec                   0.039738
nearest_natural_flow_distance_m     0.039245
traffic_avg_speed_kmh               0.037072
elevation_m                         0.025168
max_hourly_rainfall_2026_mm         0.024275
rainfall_2024_mm                    0.017555
max_hourly_rainfall_2025_mm         0.013902
natural_flow_elevation              0.012677
traffic_level_Moderate              0.012040
rainfall_2026_mm                    0.011811
max_hourly_rainfall_2024_mm         0.008695
traffic_le

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, roc_auc_score

logistic_auc = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

rf_auc = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("Logistic Regression ROC-AUC:", logistic_auc)
print("Mean:", logistic_auc.mean())

print("\nRandom Forest ROC-AUC:", rf_auc)
print("Mean:", rf_auc.mean())

Logistic Regression ROC-AUC: [0.8125 0.75   0.8125 0.875  0.9375]
Mean: 0.8375

Random Forest ROC-AUC: [0.8125 0.5625 0.9375 0.75   0.9375]
Mean: 0.8


In [ ]:
# Train final Logistic Regression on all available data
final_model = model.fit(X, y)

# Generate hotspot probability
risk_probability = final_model.predict_proba(X)[:, 1]

results = pd.DataFrame({
    "location_id": location_ids,
    "flood_label": y,
    "risk_score_pct": (risk_probability * 100).round(2),
    "predicted_class": final_model.predict(X)
})

results = results.sort_values("risk_score_pct", ascending=False)

print(results.to_string(index=False))

location_id  flood_label  risk_score_pct  predicted_class
        H02            1           99.96                1
        H01            1           99.78                1
        H04            1           99.76                1
        H03            1           99.62                1
        H05            1           99.02                1
        H16            1           96.34                1
        H17            1           96.26                1
        H19            1           94.93                1
        H18            1           94.26                1
        H14            1           90.28                1
        H20            1           90.03                1
        H12            1           87.21                1
        H11            1           84.05                1
        H06            1           75.08                1
        H09            1           74.20                1
        H10            1           72.68                1
        H13   

In [ ]:
# Create presentation risk categories
def risk_category(score):
    if score >= 75:
        return "Very High"
    elif score >= 50:
        return "High"
    elif score >= 25:
        return "Moderate"
    else:
        return "Low"

results["risk_category"] = results["risk_score_pct"].apply(risk_category)

# Save final results
results.to_csv("floodlens_risk_scores.csv", index=False)

print(results.to_string(index=False))
print("\nSaved as: floodlens_risk_scores.csv")

location_id  flood_label  risk_score_pct  predicted_class risk_category
        H02            1           99.96                1     Very High
        H01            1           99.78                1     Very High
        H04            1           99.76                1     Very High
        H03            1           99.62                1     Very High
        H05            1           99.02                1     Very High
        H16            1           96.34                1     Very High
        H17            1           96.26                1     Very High
        H19            1           94.93                1     Very High
        H18            1           94.26                1     Very High
        H14            1           90.28                1     Very High
        H20            1           90.03                1     Very High
        H12            1           87.21                1     Very High
        H11            1           84.05                1     Ve

In [ ]:
# Add coordinates to the risk results
map_results = results.merge(
    df[["location_id", "latitude", "longitude"]],
    on="location_id",
    how="left"
)

# Put coordinates near the front
map_results = map_results[
    [
        "location_id",
        "latitude",
        "longitude",
        "risk_score_pct",
        "risk_category",
        "flood_label",
        "predicted_class"
    ]
]

# Save map-ready file
map_results.to_csv("floodlens_map_ready.csv", index=False)

print(map_results.to_string(index=False))
print("\nSaved as: floodlens_map_ready.csv")

location_id  latitude  longitude  risk_score_pct risk_category  flood_label  predicted_class
        H02 28.444206  76.999750           99.96     Very High            1                1
        H01 28.438870  77.005661           99.78     Very High            1                1
        H04 28.460984  77.042014           99.76     Very High            1                1
        H03 28.454574  77.042674           99.62     Very High            1                1
        H05 28.453822  77.045449           99.02     Very High            1                1
        H16 28.501758  77.067002           96.34     Very High            1                1
        H17 28.512149  77.067662           96.26     Very High            1                1
        H19 28.508526  77.066475           94.93     Very High            1                1
        H18 28.501930  77.061334           94.26     Very High            1                1
        H14 28.492295  77.058738           90.28     Very High        

In [ ]:
import folium

# Center map on Gurugram
m = folium.Map(
    location=[map_results["latitude"].mean(), map_results["longitude"].mean()],
    zoom_start=12
)

# Risk colors
risk_colors = {
    "Very High": "red",
    "High": "orange",
    "Moderate": "yellow",
    "Low": "green"
}

# Add locations
for _, row in map_results.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=8,
        color=risk_colors[row["risk_category"]],
        fill=True,
        fill_color=risk_colors[row["risk_category"]],
        fill_opacity=0.75,
        popup=(
            f"<b>{row['location_id']}</b><br>"
            f"Risk Score: {row['risk_score_pct']}%<br>"
            f"Risk Category: {row['risk_category']}<br>"
            f"Known Hotspot Label: {row['flood_label']}"
        )
    ).add_to(m)

# Display
m

In [ ]:
# Save the FloodLens map
m.save("floodlens_gurugram_risk_map.html")

print("Map saved successfully!")

Map saved successfully!


In [ ]:
import joblib

# Save trained model
joblib.dump(final_model, "floodlens_model.pkl")

# Save the exact feature order used by the model
joblib.dump(list(X.columns), "floodlens_feature_columns.pkl")

print("Model saved: floodlens_model.pkl")
print("Feature list saved: floodlens_feature_columns.pkl")

Model saved: floodlens_model.pkl
Feature list saved: floodlens_feature_columns.pkl


In [ ]:
def predict_flood_risk(input_data):
    """
    input_data: dictionary containing the 24 model features
    """

    # Keep exactly the same feature order as training
    input_df = pd.DataFrame([input_data])[X.columns]

    probability = final_model.predict_proba(input_df)[0, 1]
    score = round(probability * 100, 2)

    if score >= 75:
        category = "Very High"
    elif score >= 50:
        category = "High"
    elif score >= 25:
        category = "Moderate"
    else:
        category = "Low"

    return {
        "risk_score_pct": score,
        "risk_category": category
    }

print("Prediction function ready!")

Prediction function ready!


In [ ]:
import joblib

loaded_model = joblib.load("floodlens_model.pkl")
loaded_features = joblib.load("floodlens_feature_columns.pkl")

print("Model loaded:", type(loaded_model).__name__)
print("Number of features:", len(loaded_features))
print("Features:", loaded_features)

Model loaded: Pipeline
Number of features: 24
Features: ['latitude', 'longitude', 'elevation_m', 'rainfall_2026_mm', 'max_hourly_rainfall_2026_mm', 'rainfall_2024_mm', 'max_hourly_rainfall_2024_mm', 'rainfall_2025_mm', 'max_hourly_rainfall_2025_mm', 'route_distance_m', 'normal_duration_sec', 'traffic_duration_sec', 'traffic_delay_sec', 'congestion_ratio', 'traffic_avg_speed_kmh', 'distance_to_watershed_boundary_m', 'natural_flow_elevation', 'natural_flow_in_flow', 'natural_flow_out_flow', 'drain_area_sq_km', 'nearest_natural_flow_distance_m', 'traffic_level_High', 'traffic_level_Moderate', 'traffic_level_Severe']


In [ ]:
# Test the independently loaded model on H01
test_row = df[df["location_id"] == "H01"].copy()

X_test_location = test_row[loaded_features]

probability = loaded_model.predict_proba(X_test_location)[0, 1]
prediction = loaded_model.predict(X_test_location)[0]

print("Location: H01")
print("Risk Score:", round(probability * 100, 2), "%")
print("Predicted Class:", prediction)

Location: H01
Risk Score: 99.78 %
Predicted Class: 1


In [ ]:
import zipfile

files_to_package = [
    "floodlens_model.pkl",
    "floodlens_feature_columns.pkl",
    "floodlens_map_ready.csv",
    "floodlens_risk_scores.csv",
    "floodlens_gurugram_risk_map.html"
]

with zipfile.ZipFile("FloodLens_model_package.zip", "w") as zipf:
    for file in files_to_package:
        zipf.write(file)

print("Package created: FloodLens_model_package.zip")

from google.colab import files
files.download("FloodLens_model_package.zip")

Package created: FloodLens_model_package.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import zipfile
import os

with zipfile.ZipFile("FloodLens_model_package.zip", "r") as zip_ref:
    zip_ref.extractall("jalnetra_model")

print("Extracted files:")
print(os.listdir("jalnetra_model"))

Extracted files:
['floodlens_gurugram_risk_map.html', 'floodlens_map_ready.csv', 'floodlens_model.pkl', 'floodlens_feature_columns.pkl', 'floodlens_risk_scores.csv']


In [ ]:
import os
import time
from google.colab import output

os.system("pkill -f 'streamlit run app.py' || true")
time.sleep(2)

os.system(
    "streamlit run app.py "
    "--server.port 8501 "
    "--server.address 0.0.0.0 "
    "--server.enableCORS false "
    "--server.enableXsrfProtection false "
    "> streamlit.log 2>&1 &"
)

time.sleep(5)

url = output.eval_js("google.colab.kernel.proxyPort(8501)")
print("🌧️ JalNetra is LIVE!")
print(url)

🌧️ JalNetra is LIVE!
https://8501-m-s-kkb-usw1b2-ksrun3tdnzct-b.us-west1-2.prod.colab.dev


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

tuned_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

param_grid = {
    "classifier__C": [0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10, 100],
    "classifier__penalty": ["l2"]
}

grid = GridSearchCV(
    tuned_model,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Best Parameters:", grid.best_params_)
print("CV Accuracy:", round(grid.best_score_, 4))
print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

Best Parameters: {'classifier__C': 5, 'classifier__penalty': 'l2'}
CV Accuracy: 0.7857
Test Accuracy: 0.75
Test ROC-AUC: 1.0


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score
import numpy as np

# Use the tuned model settings
threshold_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=grid.best_params_["classifier__C"],
        penalty="l2",
        max_iter=5000,
        random_state=42
    ))
])

# Get out-of-fold probabilities from TRAINING data only
cv_threshold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_oof_prob = cross_val_predict(
    threshold_model,
    X_train,
    y_train,
    cv=cv_threshold,
    method="predict_proba"
)[:, 1]

# Find the best threshold using training data only
thresholds = np.arange(0.30, 0.71, 0.01)

threshold_results = []

for t in thresholds:
    pred = (train_oof_prob >= t).astype(int)
    acc = accuracy_score(y_train, pred)
    threshold_results.append((t, acc))

best_threshold, best_cv_acc = max(
    threshold_results,
    key=lambda x: x[1]
)

# Refit on all training data
threshold_model.fit(X_train, y_train)

# Evaluate ONCE on untouched test data
test_prob = threshold_model.predict_proba(X_test)[:, 1]
y_test_threshold_pred = (test_prob >= best_threshold).astype(int)

test_accuracy = accuracy_score(y_test, y_test_threshold_pred)

print("Best threshold:", round(best_threshold, 2))
print("Training CV accuracy at threshold:", round(best_cv_acc, 4))
print("Test accuracy:", round(test_accuracy, 4))

print("\nTest probabilities:")
for prob, actual, pred in zip(test_prob, y_test, y_test_threshold_pred):
    print(
        f"Actual={actual}  "
        f"Probability={prob:.3f}  "
        f"Prediction={pred}"
    )

Best threshold: 0.3
Training CV accuracy at threshold: 0.75
Test accuracy: 0.75

Test probabilities:
Actual=0  Probability=0.084  Prediction=0
Actual=0  Probability=0.000  Prediction=0
Actual=1  Probability=0.994  Prediction=1
Actual=1  Probability=0.969  Prediction=1
Actual=0  Probability=0.957  Prediction=1
Actual=1  Probability=1.000  Prediction=1
Actual=0  Probability=0.892  Prediction=1
Actual=1  Probability=0.999  Prediction=1


In [ ]:
# Remove geographic coordinates
X_no_geo = X.drop(columns=["latitude", "longitude"])

cv_no_geo = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

geo_removed_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=grid.best_params_["classifier__C"],
        penalty="l2",
        max_iter=5000,
        random_state=42
    ))
])

acc_no_geo = cross_val_score(
    geo_removed_model,
    X_no_geo,
    y,
    cv=cv_no_geo,
    scoring="accuracy"
)

auc_no_geo = cross_val_score(
    geo_removed_model,
    X_no_geo,
    y,
    cv=cv_no_geo,
    scoring="roc_auc"
)

print("Without latitude/longitude")
print("5-Fold Accuracy:", acc_no_geo)
print("Mean Accuracy:", round(acc_no_geo.mean(), 4))
print("Std Dev:", round(acc_no_geo.std(), 4))

print("\n5-Fold ROC-AUC:", auc_no_geo)
print("Mean ROC-AUC:", round(auc_no_geo.mean(), 4))

Without latitude/longitude
5-Fold Accuracy: [0.75  0.625 0.75  0.75  0.875]
Mean Accuracy: 0.75
Std Dev: 0.0791

5-Fold ROC-AUC: [0.8125 0.6875 0.6875 1.     0.875 ]
Mean ROC-AUC: 0.8125


In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=5,
            penalty="l2",
            max_iter=5000,
            random_state=42
        ))
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            C=1,
            kernel="rbf",
            probability=True,
            random_state=42
        ))
    ]),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.03,
        max_depth=2,
        random_state=42
    )
}

for name, candidate in models.items():
    scores = cross_val_score(
        candidate,
        X_no_geo,
        y,
        cv=cv_no_geo,
        scoring="accuracy"
    )

    auc_scores = cross_val_score(
        candidate,
        X_no_geo,
        y,
        cv=cv_no_geo,
        scoring="roc_auc"
    )

    print(f"\n{name}")
    print("Accuracy:", scores)
    print("Mean Accuracy:", round(scores.mean(), 4))
    print("ROC-AUC:", round(auc_scores.mean(), 4))


Logistic Regression
Accuracy: [0.75  0.625 0.75  0.75  0.875]
Mean Accuracy: 0.75
ROC-AUC: 0.8125

SVM
Accuracy: [0.625 0.625 0.875 0.75  0.5  ]
Mean Accuracy: 0.675
ROC-AUC: 0.85

Gradient Boosting
Accuracy: [0.5  0.5  0.75 0.75 0.75]
Mean Accuracy: 0.65
ROC-AUC: 0.6875


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

l1_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

param_grid_l1 = {
    "classifier__C": [0.001, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1, 2, 5, 10]
}

grid_l1 = GridSearchCV(
    l1_model,
    param_grid_l1,
    cv=cv_no_geo,
    scoring="accuracy"
)

grid_l1.fit(X_no_geo, y)

print("Best L1 C:", grid_l1.best_params_)
print("Best CV Accuracy:", round(grid_l1.best_score_, 4))

# Show features retained by L1
coef = grid_l1.best_estimator_.named_steps["classifier"].coef_[0]

selected_features = [
    feature for feature, value in zip(X_no_geo.columns, coef)
    if abs(value) > 1e-8
]

print("\nSelected features:")
print(selected_features)
print("Number selected:", len(selected_features))

Best L1 C: {'classifier__C': 0.5}
Best CV Accuracy: 0.775

Selected features:
['max_hourly_rainfall_2026_mm', 'rainfall_2025_mm', 'normal_duration_sec', 'distance_to_watershed_boundary_m', 'natural_flow_elevation', 'natural_flow_in_flow']
Number selected: 6


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# Remove latitude/longitude from train and test
X_train_no_geo = X_train.drop(columns=["latitude", "longitude"])
X_test_no_geo = X_test.drop(columns=["latitude", "longitude"])

# L1 Logistic Regression
fair_l1_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

l1_grid = GridSearchCV(
    fair_l1_model,
    {
        "classifier__C": [
            0.001, 0.01, 0.03, 0.05, 0.1,
            0.3, 0.5, 1, 2, 5, 10
        ]
    },
    cv=5,
    scoring="accuracy"
)

# IMPORTANT: fit only on training data
l1_grid.fit(X_train_no_geo, y_train)

best_l1 = l1_grid.best_estimator_

# Evaluate on untouched test set
test_pred = best_l1.predict(X_test_no_geo)
test_prob = best_l1.predict_proba(X_test_no_geo)[:, 1]

print("Best C:", l1_grid.best_params_["classifier__C"])
print("Training CV Accuracy:", round(l1_grid.best_score_, 4))
print("Test Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

coef = best_l1.named_steps["classifier"].coef_[0]

selected = [
    feature
    for feature, value in zip(X_train_no_geo.columns, coef)
    if abs(value) > 1e-8
]

print("\nSelected Features:")
print(selected)

Best C: 2
Training CV Accuracy: 0.8476
Test Accuracy: 0.875
Test ROC-AUC: 1.0

Confusion Matrix:
[[3 1]
 [0 4]]

Selected Features:
['elevation_m', 'rainfall_2026_mm', 'max_hourly_rainfall_2026_mm', 'rainfall_2024_mm', 'max_hourly_rainfall_2024_mm', 'rainfall_2025_mm', 'normal_duration_sec', 'distance_to_watershed_boundary_m', 'natural_flow_elevation', 'natural_flow_in_flow', 'drain_area_sq_km']


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

final_l1_cv_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=2,
        max_iter=5000,
        random_state=42
    ))
])

X_model = X.drop(columns=["latitude", "longitude"])

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

acc_scores = cross_val_score(
    final_l1_cv_model,
    X_model,
    y,
    cv=repeated_cv,
    scoring="accuracy"
)

auc_scores = cross_val_score(
    final_l1_cv_model,
    X_model,
    y,
    cv=repeated_cv,
    scoring="roc_auc"
)

print("Repeated 5-Fold CV — 10 repeats")
print("Accuracy Mean:", round(acc_scores.mean(), 4))
print("Accuracy Std:", round(acc_scores.std(), 4))
print("Accuracy Min:", round(acc_scores.min(), 4))
print("Accuracy Max:", round(acc_scores.max(), 4))

print("\nROC-AUC Mean:", round(auc_scores.mean(), 4))
print("ROC-AUC Std:", round(auc_scores.std(), 4))

Repeated 5-Fold CV — 10 repeats
Accuracy Mean: 0.7625
Accuracy Std: 0.1231
Accuracy Min: 0.5
Accuracy Max: 1.0

ROC-AUC Mean: 0.8538
ROC-AUC Std: 0.135


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_model = X.drop(columns=["latitude", "longitude"])

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

configs = [
    ("L1 C=0.1", "l1", 0.1),
    ("L1 C=0.3", "l1", 0.3),
    ("L1 C=0.5", "l1", 0.5),
    ("L1 C=1",   "l1", 1),
    ("L1 C=2",   "l1", 2),
    ("L1 C=5",   "l1", 5),
    ("L2 C=0.1", "l2", 0.1),
    ("L2 C=0.5", "l2", 0.5),
    ("L2 C=1",   "l2", 1),
    ("L2 C=2",   "l2", 2),
    ("L2 C=5",   "l2", 5),
]

results_compare = []

for name, penalty, C in configs:

    model_test = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            penalty=penalty,
            C=C,
            solver="liblinear",
            max_iter=5000,
            random_state=42
        ))
    ])

    acc = cross_val_score(
        model_test,
        X_model,
        y,
        cv=repeated_cv,
        scoring="accuracy"
    )

    auc = cross_val_score(
        model_test,
        X_model,
        y,
        cv=repeated_cv,
        scoring="roc_auc"
    )

    results_compare.append({
        "Model": name,
        "Accuracy": acc.mean(),
        "Accuracy_Std": acc.std(),
        "ROC_AUC": auc.mean()
    })

comparison_df = pd.DataFrame(results_compare).sort_values(
    "Accuracy",
    ascending=False
)

print(comparison_df.round(4).to_string(index=False))

   Model  Accuracy  Accuracy_Std  ROC_AUC
L1 C=0.5    0.7725        0.1316   0.8688
  L1 C=5    0.7675        0.1347   0.8550
  L1 C=2    0.7625        0.1231   0.8538
  L2 C=5    0.7625        0.1305   0.8450
  L1 C=1    0.7575        0.1379   0.8550
  L2 C=1    0.7500        0.1299   0.8462
L2 C=0.5    0.7475        0.1425   0.8425
  L2 C=2    0.7450        0.1322   0.8462
L1 C=0.3    0.7250        0.1392   0.8412
L2 C=0.1    0.7200        0.1532   0.8338
L1 C=0.1    0.4950        0.0245   0.4931


In [ ]:
import numpy as np

# Start from the non-geographic features
X_engineered = X.drop(columns=["latitude", "longitude"]).copy()

# Historical rainfall baselines
rain_hist_mean = (
    X_engineered["rainfall_2024_mm"] +
    X_engineered["rainfall_2025_mm"]
) / 2

hourly_hist_mean = (
    X_engineered["max_hourly_rainfall_2024_mm"] +
    X_engineered["max_hourly_rainfall_2025_mm"]
) / 2

# 1. 2026 rainfall anomaly vs previous two years
X_engineered["rainfall_2026_vs_history"] = (
    X_engineered["rainfall_2026_mm"] /
    rain_hist_mean.replace(0, np.nan)
).fillna(0)

# 2. 2026 peak-hour rainfall anomaly
X_engineered["hourly_2026_vs_history"] = (
    X_engineered["max_hourly_rainfall_2026_mm"] /
    hourly_hist_mean.replace(0, np.nan)
).fillna(0)

# 3. Natural flow balance
X_engineered["flow_balance"] = (
    X_engineered["natural_flow_out_flow"] -
    X_engineered["natural_flow_in_flow"]
)

# 4. Traffic delay fraction
X_engineered["traffic_delay_fraction"] = (
    X_engineered["traffic_delay_sec"] /
    X_engineered["normal_duration_sec"].replace(0, np.nan)
).fillna(0)

print("Original features:", X.shape[1])
print("Engineered features:", X_engineered.shape[1])

print("\nNew features:")
print([
    "rainfall_2026_vs_history",
    "hourly_2026_vs_history",
    "flow_balance",
    "traffic_delay_fraction"
])

Original features: 24
Engineered features: 26

New features:
['rainfall_2026_vs_history', 'hourly_2026_vs_history', 'flow_balance', 'traffic_delay_fraction']


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

engineered_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        C=0.5,
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

repeated_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

acc_engineered = cross_val_score(
    engineered_model,
    X_engineered,
    y,
    cv=repeated_cv,
    scoring="accuracy"
)

auc_engineered = cross_val_score(
    engineered_model,
    X_engineered,
    y,
    cv=repeated_cv,
    scoring="roc_auc"
)

print("Engineered Features — 5-Fold × 10 Repeats")
print("Accuracy Mean:", round(acc_engineered.mean(), 4))
print("Accuracy Std:", round(acc_engineered.std(), 4))
print("ROC-AUC Mean:", round(auc_engineered.mean(), 4))
print("ROC-AUC Std:", round(auc_engineered.std(), 4))

Engineered Features — 5-Fold × 10 Repeats
Accuracy Mean: 0.7525
Accuracy Std: 0.1447
ROC-AUC Mean: 0.8438
ROC-AUC Std: 0.1475


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selection_results = []

for k in [5, 7, 9, 11, 13, 15, 18, 20]:

    selection_model = Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=k)),
        ("classifier", LogisticRegression(
            penalty="l2",
            C=1,
            max_iter=5000,
            random_state=42
        ))
    ])

    scores = cross_val_score(
        selection_model,
        X_model,
        y,
        cv=repeated_cv,
        scoring="accuracy"
    )

    auc = cross_val_score(
        selection_model,
        X_model,
        y,
        cv=repeated_cv,
        scoring="roc_auc"
    )

    selection_results.append({
        "k_features": k,
        "accuracy_mean": scores.mean(),
        "accuracy_std": scores.std(),
        "roc_auc_mean": auc.mean()
    })

selection_df = pd.DataFrame(selection_results)

print(selection_df.round(4).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [21] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [21] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [21] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/

 k_features  accuracy_mean  accuracy_std  roc_auc_mean
          5         0.7075        0.1428        0.7612
          7         0.6850        0.1397        0.7550
          9         0.6800        0.1661        0.7562
         11         0.6700        0.1495        0.7538
         13         0.6750        0.1581        0.7688
         15         0.7000        0.1768        0.7837
         18         0.7300        0.1646        0.8338
         20         0.7375        0.1420        0.8400


In [ ]:
# Remove traffic-level dummy variables
traffic_dummy_cols = [
    "traffic_level_High",
    "traffic_level_Moderate",
    "traffic_level_Severe"
]

X_no_traffic_dummies = X_model.drop(columns=traffic_dummy_cols)

traffic_clean_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        C=0.5,
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

clean_acc = cross_val_score(
    traffic_clean_model,
    X_no_traffic_dummies,
    y,
    cv=repeated_cv,
    scoring="accuracy"
)

clean_auc = cross_val_score(
    traffic_clean_model,
    X_no_traffic_dummies,
    y,
    cv=repeated_cv,
    scoring="roc_auc"
)

print("Without traffic-level dummy variables")
print("Accuracy Mean:", round(clean_acc.mean(), 4))
print("Accuracy Std:", round(clean_acc.std(), 4))
print("ROC-AUC Mean:", round(clean_auc.mean(), 4))
print("ROC-AUC Std:", round(clean_auc.std(), 4))

Without traffic-level dummy variables
Accuracy Mean: 0.7825
Accuracy Std: 0.141
ROC-AUC Mean: 0.8725
ROC-AUC Std: 0.1281


In [ ]:
# Final JalNetra model
final_jalnetra_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        C=0.5,
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

# Use cleaned features:
# - no latitude/longitude
# - no traffic-level dummy variables
X_final = X_model.drop(columns=[
    "traffic_level_High",
    "traffic_level_Moderate",
    "traffic_level_Severe"
])

# Train on all 40 locations
final_jalnetra_model.fit(X_final, y)

# Save model + exact feature order
joblib.dump(final_jalnetra_model, "jalnetra_final_model.pkl")
joblib.dump(list(X_final.columns), "jalnetra_final_features.pkl")

print("✅ Final JalNetra model trained")
print("Features:", len(X_final.columns))
print("Saved:")
print("  jalnetra_final_model.pkl")
print("  jalnetra_final_features.pkl")

✅ Final JalNetra model trained
Features: 19
Saved:
  jalnetra_final_model.pkl
  jalnetra_final_features.pkl


In [ ]:
print("Final model features:")
print(X_final.columns.tolist())
print("\nNumber of features:", len(X_final.columns))

Final model features:
['elevation_m', 'rainfall_2026_mm', 'max_hourly_rainfall_2026_mm', 'rainfall_2024_mm', 'max_hourly_rainfall_2024_mm', 'rainfall_2025_mm', 'max_hourly_rainfall_2025_mm', 'route_distance_m', 'normal_duration_sec', 'traffic_duration_sec', 'traffic_delay_sec', 'congestion_ratio', 'traffic_avg_speed_kmh', 'distance_to_watershed_boundary_m', 'natural_flow_elevation', 'natural_flow_in_flow', 'natural_flow_out_flow', 'drain_area_sq_km', 'nearest_natural_flow_distance_m']

Number of features: 19


In [ ]:
# Create two synthetic sample profiles from the real dataset:
# 1. Control-like profile = median of known control locations
# 2. Hotspot-like profile = median of known hotspot locations

control_profile = X_final[y == 0].median()
hotspot_profile = X_final[y == 1].median()

sample_inputs = pd.DataFrame([
    control_profile,
    hotspot_profile
], index=["Sample Control-like", "Sample Hotspot-like"])

# Predict using the final JalNetra model
sample_probabilities = final_jalnetra_model.predict_proba(sample_inputs)[:, 1]

for name, prob in zip(sample_inputs.index, sample_probabilities):
    score = round(prob * 100, 2)

    if score >= 75:
        category = "Very High"
    elif score >= 50:
        category = "High"
    elif score >= 25:
        category = "Moderate"
    else:
        category = "Low"

    print(f"{name}")
    print(f"Risk Score: {score}%")
    print(f"Risk Category: {category}")
    print("-" * 40)

Sample Control-like
Risk Score: 30.66%
Risk Category: Moderate
----------------------------------------
Sample Hotspot-like
Risk Score: 77.97%
Risk Category: Very High
----------------------------------------


In [ ]:
# Manual sample: based on the observed feature ranges
manual_input = {
    "elevation_m": 220,
    "rainfall_2026_mm": 180,
    "max_hourly_rainfall_2026_mm": 45,
    "rainfall_2024_mm": 165,
    "max_hourly_rainfall_2024_mm": 42,
    "rainfall_2025_mm": 175,
    "max_hourly_rainfall_2025_mm": 44,
    "route_distance_m": 4200,
    "normal_duration_sec": 600,
    "traffic_duration_sec": 900,
    "traffic_delay_sec": 300,
    "congestion_ratio": 1.50,
    "traffic_avg_speed_kmh": 22,
    "distance_to_watershed_boundary_m": 350,
    "natural_flow_elevation": 215,
    "natural_flow_in_flow": 1.2,
    "natural_flow_out_flow": 0.7,
    "drain_area_sq_km": 0.8,
    "nearest_natural_flow_distance_m": 120
}

manual_df = pd.DataFrame([manual_input])[X_final.columns]

probability = final_jalnetra_model.predict_proba(manual_df)[0, 1]
score = round(probability * 100, 2)

if score >= 75:
    category = "Very High"
elif score >= 50:
    category = "High"
elif score >= 25:
    category = "Moderate"
else:
    category = "Low"

print("JalNetra Sample Prediction")
print("Risk Score:", score, "%")
print("Risk Category:", category)

JalNetra Sample Prediction
Risk Score: 100.0 %
Risk Category: Very High


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score

# Out-of-fold predictions for every real location
oof_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        C=0.5,
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

oof_prob = cross_val_predict(
    oof_model,
    X_final,
    y,
    cv=oof_cv,
    method="predict_proba"
)[:, 1]

oof_pred = (oof_prob >= 0.5).astype(int)

oof_results = pd.DataFrame({
    "location_id": df["location_id"],
    "actual": y.values,
    "risk_score_pct": (oof_prob * 100).round(2),
    "predicted": oof_pred
})

print("Out-of-fold accuracy:",
      round(accuracy_score(y, oof_pred), 4))

print("\nSample predictions:")
print(oof_results.sort_values("risk_score_pct", ascending=False).head(10).to_string(index=False))

Out-of-fold accuracy: 0.775

Sample predictions:
location_id  actual  risk_score_pct  predicted
        H01       1           96.06          1
        H04       1           95.75          1
        H02       1           95.09          1
        H03       1           93.95          1
        H17       1           91.71          1
        H05       1           90.09          1
        H16       1           82.60          1
        C05       0           78.15          1
        H19       1           76.65          1
        H18       1           74.88          1


In [ ]:
# Train the final improved JalNetra model on all 40 locations

final_jalnetra_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1",
        C=0.5,
        solver="liblinear",
        max_iter=5000,
        random_state=42
    ))
])

# 19 final features:
# no latitude/longitude
# no traffic-level dummy variables
X_final = X_model.drop(columns=[
    "traffic_level_High",
    "traffic_level_Moderate",
    "traffic_level_Severe"
])

final_jalnetra_model.fit(X_final, y)

# Save exact model and feature order
joblib.dump(
    final_jalnetra_model,
    "jalnetra_final_model.pkl"
)

joblib.dump(
    list(X_final.columns),
    "jalnetra_final_features.pkl"
)

print("✅ Improved JalNetra model saved")
print("Features:", len(X_final.columns))
print("Cross-validated accuracy: ~78.25%")
print("Cross-validated ROC-AUC: ~87.25%")

✅ Improved JalNetra model saved
Features: 19
Cross-validated accuracy: ~78.25%
Cross-validated ROC-AUC: ~87.25%


In [ ]:
# Generate updated JalNetra risk scores using the improved model

final_prob = final_jalnetra_model.predict_proba(X_final)[:, 1]
final_pred = final_jalnetra_model.predict(X_final)

updated_results = pd.DataFrame({
    "location_id": df["location_id"],
    "flood_label": y.values,
    "risk_score_pct": (final_prob * 100).round(2),
    "predicted_class": final_pred
})

def risk_category(score):
    if score >= 75:
        return "Very High"
    elif score >= 50:
        return "High"
    elif score >= 25:
        return "Moderate"
    else:
        return "Low"

updated_results["risk_category"] = (
    updated_results["risk_score_pct"].apply(risk_category)
)

updated_results = updated_results.sort_values(
    "risk_score_pct",
    ascending=False
)

updated_results.to_csv(
    "jalnetra_final_risk_scores.csv",
    index=False
)

print(updated_results.to_string(index=False))
print("\n✅ Saved: jalnetra_final_risk_scores.csv")

location_id  flood_label  risk_score_pct  predicted_class risk_category
        H02            1           97.69                1     Very High
        H01            1           97.35                1     Very High
        H04            1           95.84                1     Very High
        H03            1           95.02                1     Very High
        H05            1           91.67                1     Very High
        H16            1           84.55                1     Very High
        H17            1           83.99                1     Very High
        H20            1           83.68                1     Very High
        H19            1           81.46                1     Very High
        H18            1           80.97                1     Very High
        H14            1           75.98                1     Very High
        H12            1           67.23                1          High
        H13            1           65.99                1       

In [ ]:
# Add coordinates to the NEW JalNetra risk scores

jalnetra_map = updated_results.merge(
    df[["location_id", "latitude", "longitude"]],
    on="location_id",
    how="left"
)

jalnetra_map = jalnetra_map[
    [
        "location_id",
        "latitude",
        "longitude",
        "risk_score_pct",
        "risk_category",
        "flood_label",
        "predicted_class"
    ]
]

jalnetra_map.to_csv(
    "jalnetra_final_map_ready.csv",
    index=False
)

print(jalnetra_map.to_string(index=False))
print("\n✅ Saved: jalnetra_final_map_ready.csv")

location_id  latitude  longitude  risk_score_pct risk_category  flood_label  predicted_class
        H02 28.444206  76.999750           97.69     Very High            1                1
        H01 28.438870  77.005661           97.35     Very High            1                1
        H04 28.460984  77.042014           95.84     Very High            1                1
        H03 28.454574  77.042674           95.02     Very High            1                1
        H05 28.453822  77.045449           91.67     Very High            1                1
        H16 28.501758  77.067002           84.55     Very High            1                1
        H17 28.512149  77.067662           83.99     Very High            1                1
        H20 28.504993  77.037670           83.68     Very High            1                1
        H19 28.508526  77.066475           81.46     Very High            1                1
        H18 28.501930  77.061334           80.97     Very High        

In [ ]:
import folium
import zipfile
import os

# Create updated JalNetra map
m_final = folium.Map(
    location=[
        jalnetra_map["latitude"].mean(),
        jalnetra_map["longitude"].mean()
    ],
    zoom_start=12
)

risk_colors = {
    "Very High": "red",
    "High": "orange",
    "Moderate": "yellow",
    "Low": "green"
}

for _, row in jalnetra_map.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=8,
        color=risk_colors[row["risk_category"]],
        fill=True,
        fill_color=risk_colors[row["risk_category"]],
        fill_opacity=0.75,
        popup=(
            f"<b>{row['location_id']}</b><br>"
            f"Risk Score: {row['risk_score_pct']}%<br>"
            f"Risk Category: {row['risk_category']}<br>"
            f"Known Hotspot Label: {row['flood_label']}"
        )
    ).add_to(m_final)

m_final.save("jalnetra_final_map.html")

# Package improved model files
package_files = [
    "jalnetra_final_model.pkl",
    "jalnetra_final_features.pkl",
    "jalnetra_final_risk_scores.csv",
    "jalnetra_final_map_ready.csv",
    "jalnetra_final_map.html"
]

with zipfile.ZipFile("JalNetra_FINAL_PACKAGE.zip", "w") as z:
    for file in package_files:
        z.write(file)

print("✅ JalNetra final package created")
print("JalNetra_FINAL_PACKAGE.zip")

from google.colab import files
files.download("JalNetra_FINAL_PACKAGE.zip")

✅ JalNetra final package created
JalNetra_FINAL_PACKAGE.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

events = [
    # date, location/area, rainfall_mm, actual_waterlogging, match_type, source
    ["2024-06-28", "Sector 10A", 30.0, 1, "sector", "Social News XYZ"],
    ["2024-06-28", "Sector 31", 30.0, 1, "sector", "Social News XYZ"],
    ["2024-06-28", "Palam Vihar", 30.0, 1, "area", "Social News XYZ"],

    ["2024-07-04", "Palam Vihar", 35.5, 1, "area", "Hindustan Times"],

    ["2024-07-26", "Sector 57", 2.5, 1, "sector", "Hindustan Times"],

    ["2024-08-11", "Sector 15", 20.5, 1, "sector", "Times of India"],
    ["2024-08-11", "Sector 31", 20.5, 1, "sector", "Times of India"],

    ["2024-08-17", "Palam Vihar C Block", None, 1, "block", "Hindustan Times"],
    ["2024-08-17", "Palam Vihar D Block", None, 1, "block", "Hindustan Times"],

    ["2022-05-25", "Sector 15", None, 1, "sector", "India Today"],
    ["2022-05-25", "Sector 10", None, 1, "sector", "India Today"],
    ["2022-05-25", "Palam Vihar", None, 1, "area", "India Today"],

    ["2020-07-01", "Palam Vihar A Block", None, 1, "block", "Times of India"],
]

historical_events = pd.DataFrame(
    events,
    columns=[
        "event_date",
        "event_location",
        "rainfall_mm",
        "actual_waterlogging",
        "match_type",
        "source"
    ]
)

historical_events["event_date"] = pd.to_datetime(
    historical_events["event_date"]
)

historical_events.to_csv(
    "historical_flood_events.csv",
    index=False
)

print(historical_events)
print("\nSaved:", "historical_flood_events.csv")

   event_date       event_location  rainfall_mm  actual_waterlogging  \
0  2024-06-28           Sector 10A         30.0                    1   
1  2024-06-28            Sector 31         30.0                    1   
2  2024-06-28          Palam Vihar         30.0                    1   
3  2024-07-04          Palam Vihar         35.5                    1   
4  2024-07-26            Sector 57          2.5                    1   
5  2024-08-11            Sector 15         20.5                    1   
6  2024-08-11            Sector 31         20.5                    1   
7  2024-08-17  Palam Vihar C Block          NaN                    1   
8  2024-08-17  Palam Vihar D Block          NaN                    1   
9  2022-05-25            Sector 15          NaN                    1   
10 2022-05-25            Sector 10          NaN                    1   
11 2022-05-25          Palam Vihar          NaN                    1   
12 2020-07-01  Palam Vihar A Block          NaN                 

In [ ]:
import pandas as pd

# Load your files
events = pd.read_csv("historical_flood_events.csv")
locations = pd.read_csv("master_model_ready.csv")

# Your H locations only
H = locations[locations["location_id"].str.startswith("H")].copy()

# Add a clean name column for matching
# IMPORTANT: edit these names if your actual H01-H20 names differ
location_names = {
    "H01": "Sector 10A",
    "H02": "Sector 10A",
    "H03": "Sector 15 Part II",
    "H04": "Sector 15 Part II",
    "H05": "Sector 31",
    "H06": "Sector 46",
    "H07": "Sector 57",
    "H08": "Ghata",
    "H09": "Sector 57",
    "H10": "Silokhera",
    "H11": "Nathupur",
    "H12": "Nathupur",
    "H13": "Sikanderpur",
    "H14": "Sector 18",
    "H15": "Palam Vihar C Block",
    "H16": "Palam Vihar D Block",
    "H17": "Sector 23",
    "H18": "Sector 21",
    "H19": "Sector 22B",
    "H20": "Palam Vihar A Block"
}

H["location_name"] = H["location_id"].map(location_names)

# Show the locations before matching
print("YOUR H LOCATIONS:")
print(H[["location_id", "location_name", "latitude", "longitude"]].to_string(index=False))

# Show real historical events
print("\nREAL HISTORICAL EVENTS:")
print(events.to_string(index=False))

YOUR H LOCATIONS:
location_id       location_name  latitude  longitude
        H01          Sector 10A 28.438870  77.005661
        H02          Sector 10A 28.444206  76.999750
        H03   Sector 15 Part II 28.454574  77.042674
        H04   Sector 15 Part II 28.460984  77.042014
        H05           Sector 31 28.453822  77.045449
        H06           Sector 46 28.434887  77.064065
        H07           Sector 57 28.417193  77.074298
        H08               Ghata 28.422040  77.109600
        H09           Sector 57 28.420603  77.074107
        H10           Silokhera 28.458293  77.060052
        H11            Nathupur 28.488572  77.102205
        H12            Nathupur 28.480030  77.099612
        H13         Sikanderpur 28.479154  77.094202
        H14           Sector 18 28.492295  77.058738
        H15 Palam Vihar C Block 28.481765  77.072026
        H16 Palam Vihar D Block 28.501758  77.067002
        H17           Sector 23 28.512149  77.067662
        H18           Sector

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# =========================
# 1. LOAD DATA
# =========================
data = pd.read_csv("master_model_ready.csv")
events = pd.read_csv("historical_flood_events.csv")

# =========================
# 2. TRAIN MODEL
# =========================
X = data.drop(columns=["location_id", "flood_label"])
y = data["flood_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model trained successfully.")

# =========================
# 3. MATCH REAL EVENTS
# =========================
location_names = {
    "H01": "Sector 10A",
    "H02": "Sector 10A",
    "H03": "Sector 15 Part II",
    "H04": "Sector 15 Part II",
    "H05": "Sector 31",
    "H06": "Sector 46",
    "H07": "Sector 57",
    "H08": "Ghata",
    "H09": "Sector 57",
    "H10": "Silokhera",
    "H11": "Nathupur",
    "H12": "Nathupur",
    "H13": "Sikanderpur",
    "H14": "Sector 18",
    "H15": "Palam Vihar C Block",
    "H16": "Palam Vihar D Block",
    "H17": "Sector 23",
    "H18": "Sector 21",
    "H19": "Sector 22B",
    "H20": "Palam Vihar A Block"
}

# Convert event names to the corresponding H locations
matches = []

for _, event in events.iterrows():

    name = event["event_location"]

    for location_id, location_name in location_names.items():

        # Exact/block match
        if name.lower() == location_name.lower():

            matches.append({
                "event_date": event["event_date"],
                "event_location": name,
                "location_id": location_id,
                "actual_waterlogging": event["actual_waterlogging"],
                "match_type": event["match_type"],
                "source": event["source"]
            })

        # Sector-level matching
        elif (
            event["match_type"] == "sector"
            and name.lower() == location_name.lower().replace(" part ii", "")
        ):
            matches.append({
                "event_date": event["event_date"],
                "event_location": name,
                "location_id": location_id,
                "actual_waterlogging": event["actual_waterlogging"],
                "match_type": "sector",
                "source": event["source"]
            })

matched_events = pd.DataFrame(matches)

# Remove duplicate matches
matched_events = matched_events.drop_duplicates(
    subset=["event_date", "event_location", "location_id"]
)

# =========================
# 4. PREDICT FOR MATCHED H LOCATIONS
# =========================
predictions = []

for location_id in matched_events["location_id"].unique():

    row = data[data["location_id"] == location_id]

    if len(row) == 0:
        continue

    X_location = row.drop(columns=["location_id", "flood_label"])

    prediction = model.predict(X_location)[0]
    probability = model.predict_proba(X_location)[0][1]

    predictions.append({
        "location_id": location_id,
        "model_prediction": int(prediction),
        "model_probability": round(probability, 3)
    })

predictions = pd.DataFrame(predictions)

# =========================
# 5. COMBINE ACTUAL + PREDICTION
# =========================
validation = matched_events.merge(
    predictions,
    on="location_id",
    how="left"
)

validation["correct"] = (
    validation["actual_waterlogging"]
    == validation["model_prediction"]
)

print("\n==============================")
print("REAL EVENT VALIDATION RESULTS")
print("==============================")

print(
    validation[
        [
            "event_date",
            "event_location",
            "location_id",
            "actual_waterlogging",
            "model_prediction",
            "model_probability",
            "correct",
            "source"
        ]
    ].to_string(index=False)
)

# =========================
# 6. SAVE RESULTS
# =========================
validation.to_csv(
    "real_historical_event_validation.csv",
    index=False
)

print("\nSaved: real_historical_event_validation.csv")

Model trained successfully.

REAL EVENT VALIDATION RESULTS
event_date      event_location location_id  actual_waterlogging  model_prediction  model_probability  correct          source
2024-06-28          Sector 10A         H01                    1                 1              0.640     True Social News XYZ
2024-06-28          Sector 10A         H02                    1                 1              0.733     True Social News XYZ
2024-06-28           Sector 31         H05                    1                 1              0.937     True Social News XYZ
2024-07-26           Sector 57         H07                    1                 1              0.767     True Hindustan Times
2024-07-26           Sector 57         H09                    1                 1              0.553     True Hindustan Times
2024-08-11           Sector 15         H03                    1                 1              0.967     True  Times of India
2024-08-11           Sector 15         H04                 

In [ ]:
import os

files_to_delete = [
    "historical_flood_events.csv",
    "real_historical_event_validation.csv",
    "real_historical_event_predictions.csv"
]

for file in files_to_delete:
    if os.path.exists(file):
        os.remove(file)
        print("Deleted:", file)
    else:
        print("Not found:", file)

Deleted: historical_flood_events.csv
Deleted: real_historical_event_validation.csv
Not found: real_historical_event_predictions.csv


In [ ]:
import pandas as pd

events = [
    ["2024-06-28", "Sector 10A", 30.0, 1, "sector", "Social News XYZ"],
    ["2024-06-28", "Sector 31", 30.0, 1, "sector", "Social News XYZ"],
    ["2024-06-28", "Palam Vihar", 30.0, 1, "area", "Social News XYZ"],
    ["2024-07-04", "Palam Vihar", 35.5, 1, "area", "Hindustan Times"],
    ["2024-07-26", "Sector 57", 2.5, 1, "sector", "Hindustan Times"],
    ["2024-08-11", "Sector 15", 20.5, 1, "sector", "Times of India"],
    ["2024-08-11", "Sector 31", 20.5, 1, "sector", "Times of India"],
    ["2024-08-17", "Palam Vihar C Block", None, 1, "block", "Hindustan Times"],
    ["2024-08-17", "Palam Vihar D Block", None, 1, "block", "Hindustan Times"],
    ["2022-05-25", "Sector 15", None, 1, "sector", "India Today"],
    ["2022-05-25", "Sector 10", None, 1, "sector", "India Today"],
    ["2022-05-25", "Palam Vihar", None, 1, "area", "India Today"],
    ["2020-07-01", "Palam Vihar A Block", None, 1, "block", "Times of India"],
]

events = pd.DataFrame(events, columns=[
    "event_date",
    "event_location",
    "rainfall_mm",
    "actual_waterlogging",
    "match_type",
    "source"
])

events.to_csv("historical_flood_events.csv", index=False)

print("Real historical event file recreated.")
print(events)

Real historical event file recreated.
    event_date       event_location  rainfall_mm  actual_waterlogging  \
0   2024-06-28           Sector 10A         30.0                    1   
1   2024-06-28            Sector 31         30.0                    1   
2   2024-06-28          Palam Vihar         30.0                    1   
3   2024-07-04          Palam Vihar         35.5                    1   
4   2024-07-26            Sector 57          2.5                    1   
5   2024-08-11            Sector 15         20.5                    1   
6   2024-08-11            Sector 31         20.5                    1   
7   2024-08-17  Palam Vihar C Block          NaN                    1   
8   2024-08-17  Palam Vihar D Block          NaN                    1   
9   2022-05-25            Sector 15          NaN                    1   
10  2022-05-25            Sector 10          NaN                    1   
11  2022-05-25          Palam Vihar          NaN                    1   
12  2020-07-0

In [ ]:
import pandas as pd

# Original model data
data = pd.read_csv("master_model_ready.csv")

# Real historical events
events = pd.read_csv("historical_flood_events.csv")

# H-location mapping
location_names = {
    "H01": "Sector 10A",
    "H02": "Sector 10A",
    "H03": "Sector 15 Part II",
    "H04": "Sector 15 Part II",
    "H05": "Sector 31",
    "H06": "Sector 46",
    "H07": "Sector 57",
    "H08": "Ghata",
    "H09": "Sector 57",
    "H10": "Silokhera",
    "H11": "Nathupur",
    "H12": "Nathupur",
    "H13": "Sikanderpur",
    "H14": "Sector 18",
    "H15": "Palam Vihar C Block",
    "H16": "Palam Vihar D Block",
    "H17": "Sector 23",
    "H18": "Sector 21",
    "H19": "Sector 22B",
    "H20": "Palam Vihar A Block"
}

# Match real events to H locations
matches = []

for _, event in events.iterrows():
    name = event["event_location"].lower()

    for location_id, location_name in location_names.items():
        lname = location_name.lower()

        if name == lname:
            matches.append({
                "event_date": event["event_date"],
                "event_location": event["event_location"],
                "location_id": location_id,
                "actual_waterlogging": event["actual_waterlogging"],
                "match_type": event["match_type"],
                "source": event["source"]
            })

        elif (
            event["match_type"] == "sector"
            and name == lname.replace(" part ii", "")
        ):
            matches.append({
                "event_date": event["event_date"],
                "event_location": event["event_location"],
                "location_id": location_id,
                "actual_waterlogging": event["actual_waterlogging"],
                "match_type": "sector",
                "source": event["source"]
            })

matched_events = pd.DataFrame(matches).drop_duplicates()

# ==========================================
# PREDICT USING EXISTING TRAINED MODEL
# ==========================================

results = []

for _, event in matched_events.iterrows():

    location_id = event["location_id"]

    row = data[data["location_id"] == location_id]

    if row.empty:
        continue

    X_location = row.drop(
        columns=["location_id", "flood_label"]
    )

    # EXISTING MODEL — NO TRAINING
    prediction = model.predict(X_location)[0]
    probability = model.predict_proba(X_location)[0][1]

    results.append({
        "event_date": event["event_date"],
        "event_location": event["event_location"],
        "location_id": location_id,
        "actual": int(event["actual_waterlogging"]),
        "predicted": int(prediction),
        "probability": round(probability, 3),
        "correct": int(
            event["actual_waterlogging"] == prediction
        ),
        "source": event["source"]
    })

results = pd.DataFrame(results)

print("\n======================================")
print("REAL EVENT vs EXISTING MODEL")
print("======================================\n")

print(results.to_string(index=False))

# Summary
correct = results["correct"].sum()
total = len(results)

print("\n======================================")
print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total:.2%}")
print("======================================")


REAL EVENT vs EXISTING MODEL

event_date      event_location location_id  actual  predicted  probability  correct          source
2024-06-28          Sector 10A         H01       1          1        0.640        1 Social News XYZ
2024-06-28          Sector 10A         H02       1          1        0.733        1 Social News XYZ
2024-06-28           Sector 31         H05       1          1        0.937        1 Social News XYZ
2024-07-26           Sector 57         H07       1          1        0.767        1 Hindustan Times
2024-07-26           Sector 57         H09       1          1        0.553        1 Hindustan Times
2024-08-11           Sector 15         H03       1          1        0.967        1  Times of India
2024-08-11           Sector 15         H04       1          1        0.883        1  Times of India
2024-08-11           Sector 31         H05       1          1        0.937        1  Times of India
2024-08-17 Palam Vihar C Block         H15       1          1        